# CSC 4792 MINI PROJECT

In [1]:
!pip install pandas requests beautifulsoup4 lxml pdfplumber pypdf kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 24.7 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from io import StringIO
from urllib.parse import urljoin
from urllib.parse import urlparse
import pdfplumber
import os
import re

In [3]:
pd.__version__

'2.2.3'

In [4]:
council = "Mumbwa Town council"
base_url = "https://www.mumbwacouncil.gov.zm/"

print(council)
print(base_url)

Mumbwa Town council
https://www.mumbwacouncil.gov.zm/


In [5]:
response = requests.get(base_url, timeout=30, verify=False)
print("Status code:", response.status_code)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200


In [6]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True))

Mumbwa Town Council – Mumbwa


In [7]:
links = []

for a in soup.find_all("a", href=True):
  links.append({
      "text": a.get_text(" ", strip=True),
      "url": a["href"]
  })

links_df = pd.DataFrame(links)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

links_df.head(100)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [8]:
keywords = ["administrative", "ward", "council","report", "resolution", "committee", "development", "publication", "about us", "services", "management", "departments"]

relevant_links = links_df[links_df["text"].str.lower().str.contains("|".join(keywords), na=False)]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

relevant_links

,text,url
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
10,Standing Committees,https://www.mumbwacouncil.gov.zm/?page_id=2881
12,Services,https://www.mumbwacouncil.gov.zm/?page_id=792
19,Publications,https://www.mumbwacouncil.gov.zm/?page_id=195
27,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
28,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
30,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
33,Standing Committees,https://www.mumbwacouncil.gov.zm/?page_id=2881


In [9]:
publications_url = "https://www.mumbwacouncil.gov.zm/?page_id=195"

response = requests.get(publications_url, timeout=30, verify=False)
print(response.status_code)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200


In [10]:
soup = BeautifulSoup(response.text, "html.parser")

publication_links = []
for a in soup.find_all("a", href=True):
  text = a.get_text(" ", strip=True)
  url = a["href"]

  publication_links.append({
      "document_title": text,
      "document_url": url
  })

publication_df = pd.DataFrame(publication_links)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
publication_df.head(100)

,document_title,document_url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


### council administrative data

In [11]:
administrative_keywords = [
    "administration",
    "management",
    "services",
    "department",
    "about us"
]
admin_df = publication_df[publication_df["document_title"].str.lower().str.contains("|".join(administrative_keywords), na=False)]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
admin_df

,document_title,document_url
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
12,Services,https://www.mumbwacouncil.gov.zm/?page_id=792
27,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
28,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
35,Services,https://www.mumbwacouncil.gov.zm/?page_id=792
57,"The-Public-Finance-Management-ACT-2018 – Uploaded on: December 8, 2023",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/The-Public-Finance-Management-ACT-2018.pdf
60,The-Solid-Waste-Regulation-and-Management-Act-2018 –,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/The-Solid-Waste-Regulation-and-Management-Act-2018.pdf


In [12]:
response = requests.get("https://www.mumbwacouncil.gov.zm/?page_id=2877", timeout=30, verify=False)
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text("\n", strip=True)
print(text[:5000])

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page not found – Mumbwa Town Council
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Menu
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Search
Nothing here
It looks like nothing was found at this location. Maybe try a search?
Search…
Contact Information
Mumbwa Town Council
Mumbwa District
Central Province
Tel: 260211800074
P.O. Box: 830001
Email; councilmumbwa@yahoo.com
COMPLAINTS
Quick Links
Ministry of Local Government and Rural Development
Local Government Service Comission
Ministry of Finance and Nati

In [13]:
response = requests.get("https://www.mumbwacouncil.gov.zm/?page_id=792", timeout=30, verify=False)
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text("\n", strip=True)
print(text[:5000])

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Services – Mumbwa Town Council
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Menu
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Search
The Council shall undertake the following responsibilities to ensure effective governance and public welfare within its jurisdiction:
Infrastructure and Administration
Establish and maintain offices and buildings necessary for the administration of Council affairs, as well as facilities for public meetings and assemblies.
Law, Order, and National Security
Maintain law an

In [14]:
response = requests.get("https://www.mumbwacouncil.gov.zm/?page_id=770", timeout=30, verify=False)
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text("\n", strip=True)
print(text[:5000])

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Departments – Mumbwa Town Council
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Menu
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Search
Institutional Management
Planning
Engineering Services
Finance
Human Resource and Administration
Fisheries, Livestock and Veterinary Services
Agriculture
Consists of the principle officer who in this regard is the Council Secretary, as well as procurement and audit sections.
The role of planning department is to plan the overall layout of the district, upgrading of sh

### Ward development committee information

In [15]:
ward_keywords = [
    "ward",
    "WDC",
    "stakeholder",
    "community",
    "development",
    "proposals",
    "projects",
    "policy"
]

wdc_df = publication_df[
    publication_df["document_title"].str.lower().str.contains("|".join(ward_keywords), na=False)
]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
wdc_df

,document_title,document_url
15,Projects,https://www.mumbwacouncil.gov.zm/?page_id=2244
38,Projects,https://www.mumbwacouncil.gov.zm/?page_id=2244
54,"The-Constituency-Development-Fund-Act-No.-11-of-2018 – Uploaded on: December 8, 2023",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/The-Constituency-Development-Fund-Act-No.-11-of-2018.pdf
75,"2026 CDF APPLICATIONS-COMMUNITY PROJECTS – Uploaded on: July 11, 2025",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/07/2026-CDF-APPLICATIONS-COMMUNITY-PROJECTS.pdf
92,"2024 Approved CDF Community Projects – Uploaded on: September 13, 2024",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2024/09/2024-CDF-COMMUNITY-PROJECTS-AS-AT-31ST-AUGUST-2024.pdf
93,"2024 Proposed CDF Projects All Components – Mumbwa and Nangoma Constituencies – Uploaded on: October 10, 2024",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2024/10/2024-CDF-Proposed-Projects-All-Components-Mumbwa-and-Nangoma-Constituencies.pdf
95,"2025 Minutes of Stakeholders First Monitoring Meeting for the Zambia Devolution Support Program (ZDSP) – Uploaded on: November 17, 2025",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/11/Stakeholders-Minutes-of-The-First-Monitoring-Meeting-for-the-Zambia-Devolution-Support-Program-ZDSP.pdf
96,"2025 Stakeholders-engangement-plan – Uploaded on: December 5, 2025",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/ACE-Scanner_2025_12_051.pdf
97,"2025 Minutes of Stakeholders Meeting on The Council Budget – Uploaded on: September 15, 2025",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/Minutes-of-Stake-holders-Meeting-on-The-2025-Council-Budget.pdf
98,2025 Stakeholders engagement meeting on the review of,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/2026-Stakeholders-Engangement-Meeting-on-the-2026-Budget-1.pdf


In [16]:
response = requests.get("https://www.mumbwacouncil.gov.zm/?page_id=2868", timeout=30, verify=False)
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text("\n", strip=True)
print(text[:5000])

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Civic Leaders – Mumbwa Town Council
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Menu
Home
About
Open menu
Who we are
Senior Management
Departments
Civic Leaders
Council Chairman
Nangoma
Mumbwa Central
Standing Committees
Mandate
Services
CDF
ZDSP
Projects
Open menu
Cash for work
News Updates
Media Center
Open menu
Publications
Photo Gallery
Application Forms
Contact Us
FAQs
Search
Civic Leaders
Contact Information
Mumbwa Town Council
Mumbwa District
Central Province
Tel: 260211800074
P.O. Box: 830001
Email; councilmumbwa@yahoo.com
COMPLAINTS
Quick Links
Ministry of Local Government and Rural Development
Local Government Service Comission
Ministry of Finance and National Planning
Ministry of Agriculture
National Assembly of Zambia
Local Govern

### public council resolutions

In [17]:
resolution_keywords = [
    "resolution",
    "audit",
    "report"
]

resolution_df = publication_df[
    publication_df["document_title"].str.lower().str.contains("|".join(resolution_keywords), na=False)
]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

resolution_df

,document_title,document_url
81,2025 Semi-Annual Budget Performance Report,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/Semi-Annual-Budget-Performance-Report-for-2025.pdf
115,"CDF-Report-2022 – Uploaded on: December 22, 2023",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/CDF-Report-2022.pdf
116,"Local-Government-operations-Report-2022 _- Uploaded on: December 22, 2023",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/Local-Government-operations-Report-2022_2.pdf
117,"MTC Audited Financial Statement for 2021 – Uploaded on: December 22, 2023",https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MTC-Audited-Financial-Statement-for-2021.pdf


In [18]:
response = requests.get("https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/Semi-Annual-Budget-Performance-Report-for-2025.pdf", timeout=30, verify=False)
response.raise_for_status()
with pdfplumber.open(BytesIO(response.content)) as pdf:

  for page_number, page in enumerate(pdf.pages, start=1):
    print(f"\n======== Page {page_number} ========\n")

    text = page.extract_text()
    if text: print(text)

    tables = page.extract_tables()
    for table_number, table in enumerate(tables, start=1):
      print(f"\n--- Table {table_number} ---")

      table_df = pd.DataFrame(table)
      print(table_df.to_string(index=False, header=False))


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



======== Page 1 ========

E C.;] co \ - ] o + (! N G) N Y xN SE z
F n > t U F ! ' \ f o . J l r t d , . ! H o , A ) ' o o o o p 3 o r @ o a o I I o 3 D ) a o @ t I p 3 D p o o N o a l @a z : 3 0 ) r a o o A o o ) mo a a t a a o D I ) ) l 0 I o a o o o a a o o o o a I D q o o t o o n o I a o 5g t I s r E I o F t l ( o o o ]) m a o o a p : l N Eo t I o{ t O . n 5 p o l N To Io{ ) D o a l D 5 ot o o o o c0 . o G o o p o o d 6 o e a N b o oI 3 o Io , ( a P 'E n o @ t l \ , ) i o o N - , ro @ N i , o o t o i A c r . r ) r 1 ) l o o p I 0x . ) ) o o d o o n o 6 t o o @ (r b o o o I a I D \ d o o o L o o o !o U n ! e o p p . a . o p N U A o _ I @ L a o o I D o p o a 6 o n i o A x o ) o r : p o o a E - o ! E o 4 t i o A o ) ) r F o o o o t g C 8 S 5 H U !.H , J ' $ t z E a E ! E t , ,> ' ' - F d > O 4 l € 6 F
' H , ! o ! o , ' o p . o p o p o il @ I a p D E E n ' E F
N i;...: o
R o3 3\ E:d'
F
B
o. Fl
z
o
E
. . o - oI o o 6 < o o t t ) ) t ( @ @ o Jl " . @ ! O { - {( + . , I o \ C ( t -. o . n

In [19]:
response = requests.get("https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/CDF-Report-2022.pdf", timeout=30, verify=False)
response.raise_for_status()
with pdfplumber.open(BytesIO(response.content)) as pdf:
  page = pdf.pages[266]
  text = page.extract_text()
  if text: print(text)

  tables = page.extract_tables()
  for table_number, table in enumerate(tables, start=1):
    print(f"\n--- Table {table_number} ---")

    table_df = pd.DataFrame(table)
    print(table_df.to_string)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


80. Mumbwa Town Council
a. Mumbwa Central Constituency
i. Soft Loans - Failure to Collect Loan Repayments
According to the CDF Guidelines on Loan Repayment b (i) states that the first
payment shall be 60 days from the date of getting the loan; (ii) the subsequent
instalments shall be paid at the end of each month.
During the period under review, sixty (60) applicants were selected and awarded
soft loans in amounts totalling K2,557,090 to undertake various projects.
However, it was observed that the sixty (60) beneficiaries were required to pay
monthly amounts totalling K112,336 with repayment periods ranging from
twelve (12) to thirty six (36) months.
Contrary to the guidelines, 31st October 2023, the Council only collected
K78,525 as loan repayment for the months of August and September 2023
leaving a balance of K149,147. See table 1 below.
Table 1: Uncollected Loan Repayment
248
ME
AST
n
ueo
o n t h
d in g
g u s t
p te m
t a l
b e r
L o a n A
E x p e
K
1
1
2
mc
112
o u
t e d
2 ,3
2 ,

In [ ]:
response = requests.get("https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/Local-Government-operations-Report-2022_2.pdf", timeout=30, verify=False)
response.raise_for_status()
with pdfplumber.open(BytesIO(response.content)) as pdf:
  page = pdf.pages[186]
  text = page.extract_text()
  if text: print(text)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


81. Mumbwa Town Council
a. Unapproved Payments
Section 54 (1) (a) (b) of the Public Finance Management (General) Regulations, 2020
states that, “where a hard copy payment voucher is used, the original of a payment
voucher shall be signed by a warrant holder, sub warrant holder or by any other authorised
office holder and indicate the name and, designation of the office holder signing, and the
date below the office holder’s signature.”
Contrary to the Regulation, three (3) payment in amounts totalling K8,185 made during
the year under review, were processed without approval by a responsible officer.
82. Mushindamo Town Council
a. Failure to Collect Plot Premium
Regulation No. 9 (h) of the Public Finance Management (General) Regulations, 2020
requires the head of accounting unit of a local authority to “collect in a timely manner all
revenue and other public monies due and payable to the local authority.”
During the period under review, the Council was expected to collect amounts totalli

In [20]:
response = requests.get("https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MTC-Audited-Financial-Statement-for-2021.pdf", timeout=30, verify=False)
response.raise_for_status()
with pdfplumber.open(BytesIO(response.content)) as pdf:

  for page_number, page in enumerate(pdf.pages, start=1):
    print(f"\n======== Page {page_number} ========\n")

    text = page.extract_text()
    if text: print(text)

    tables = page.extract_tables()
    for table_number, table in enumerate(tables, start=1):
      print(f"\n--- Table {table_number} ---")

      table_df = pd.DataFrame(table)
      print(table_df.to_string(index=False, header=False))

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



======== Page 1 ========

,
y11
cou.rrunulceltous fo qe eppjessep lo
fqe 3onucrl gecr.eleA
r=f:+Zg0 Z.Dl 8000L, 1u tedll dleese bnole
lVX:+290 Z.tl gOOOLt
3ruBli: No':'""
unuq\!,econuctl@r.u16q.6on zu
conuc!IJunJuq\v/p@{eqoo 'cour
urnurgnAV IOI,AN
OOnNSl]
cl^tC
lgrrrwel'zoZE lNOfdl:NOSNCl= 33N1U3'
VASNnf
d O' 8ox 8t000t
Jqo vnpllor thnJ^tSMv
5euere1. zv\!stv
oJgco oJ}{ro Ynprlor cauorI€
:i?ffsooLt
Tnsvxr
$,H,)?:;i$X,"TSg$:# if
lf?,,T11'^f,,i,'"^oro\^NronNrrrdourHrt
uoJereuco rs ui,pe 10 crrcnrJ€ rel]aJ pnr ztr, 1A1e;cq' z\zt Mllq r'eEulps ]o ]qe q€ o^e snqloc1
ruu*or \^qorees }{ro conucrI ert rls slrpuE epodlep lqe ctueuclp siela',oul vnpllep gedop
y l c i c i o f , f p # ruE :r l(, ? , g li r 1 n' # rri T rri , "p :; { J .ro # relo : : t : o l u i )to( codres oJirro 3011orlru3 pocn,.oul es ;ebnr:ep ,, rruo
dorrcros ?vvas(oJz0r 6
eup }{ro Jreesnrr eup cruuucrp
)l(
J:Tl:irfff,:r';::J#fHH;r,,^eelruE qsJp ou rnespe{,, re,rwet,,zozt { qrcq
Dr( 3odres oSsrEuep vnplep clueuclP
e ls] sleloui

## CLEANING OF RAW DATA & DATASET CREATION
The raw data collected in previous actions is now cleaned and exported to three of the following created datasets:
- council administrative data.csv
- ward development committee information.csv
- public council resolutions.csv


In [26]:
COUNCIL_NAME = "Mumbwa Town Council"

#Functions
def clean_text(value):
    #whitespace/newlines removed and return a clean string.
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

def extract_year(value):
    #Extracting a 4-digit year from a title where possible
    match = re.search(r"\b(20\d{2})\b", str(value))
    return int(match.group(1)) if match else pd.NA

def remove_upload_text(title):
    #Removing 'Uploaded on...' text from publication titles.
    title = clean_text(title)
    title = re.sub(r"\s*[–-]\s*Uploaded on:.*$", "", title, flags=re.I)
    return title.strip(" -–")


#COUNCIL ADMINISTRATIVE DATA
admin_records = [
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Institutional Management",
        "description": "Consists of the principal officer, the Council Secretary, as well as procurement and audit sections.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Planning",
        "description": "Plans the overall layout of the district, upgrading of shanty areas, township boundaries, standard building plans, land distribution, title deeds, HIV/AIDS, gender and human rights sensitisation, community development activities, and public health.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Engineering Services",
        "description": "Facilitates feeder roads, borehole drilling, inspection of buildings, scrutiny of building plans, burial sites, fire brigade services, and maintenance of plant and machinery.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Finance",
        "description": "Collection of revenue, preparation of statutory obligations, and preparation of books of accounts.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Human Resource and Administration",
        "description": "Staff establishment, asset management, records keeping, training and development, recruitment, staff appraisals, minutes and report writing, discipline, conditions of service, and public relations services.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Fisheries, Livestock and Veterinary Services",
        "description": "",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "department",
        "category": "Agriculture",
        "description": "",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=770"
    },

    # Services extracted from the Services page
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Infrastructure and Administration",
        "description": "Establish and maintain offices, buildings, public meeting facilities and assemblies needed for Council administration.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Law, Order, and National Security",
        "description": "Maintain law and order, safeguard national security, and promote efficient governance within the Council jurisdiction.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Advertising and Public Spaces",
        "description": "Regulate and control advertisements and advertising devices visible from streets and other public areas.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Animal Welfare and Livestock Management",
        "description": "Regulate livestock movement, animal slaughter, meat certification, disposal of diseased carcasses, hygiene in abattoirs and cold storage, animal by-products, and transport of carcasses.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Transportation and Road Safety",
        "description": "Develop and maintain road infrastructure, regulate traffic and parking, and implement road-safety measures.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Public Lighting and Utilities",
        "description": "Establish and maintain street lighting systems for safety and accessibility.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Business Regulation and Licensing",
        "description": "Inspect business establishments and enforce compliance, including regulations concerning the sale and consumption of intoxicating liquor.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Housing and Urban Development",
        "description": "Construct, purchase and maintain buildings and facilities for residential accommodation and encourage adequate housing development.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    },
    {
        "council_name": COUNCIL_NAME,
        "record_type": "service",
        "category": "Land Use and Environmental Management",
        "description": "Regulate land use and building construction for public health, safety and orderly development, including demolition and removal of buildings.",
        "source_url": "https://www.mumbwacouncil.gov.zm/?page_id=792"
    }
]

council_admin_df = pd.DataFrame(admin_records)
for col in council_admin_df.select_dtypes(include="object").columns:
    council_admin_df[col] = council_admin_df[col].map(clean_text)

council_admin_df = (
    council_admin_df
    .drop_duplicates()
    .reset_index(drop=True)
)


#Ward Committe Information
wdc_pattern = (
    r"\bward\b|"
    r"\bWDC\b|"
    r"\bcommunity\b|"
    r"\bCDF\b|"
    r"\bstakeholder\b|"
    r"\bdevelopment\b|"
    r"\bproject\b"
)

wdc_info_df = publication_df.copy()
wdc_info_df["document_title"] = wdc_info_df["document_title"].map(clean_text)
wdc_info_df["document_url"] = wdc_info_df["document_url"].map(clean_text)

wdc_info_df = wdc_info_df[
    wdc_info_df["document_title"].str.contains(wdc_pattern, case=False, na=False, regex=True)
].copy()

# Keep actual downloadable/informational records, not menu links.
wdc_info_df = wdc_info_df[
    wdc_info_df["document_url"].str.contains(r"\.pdf($|\?)", case=False, na=False, regex=True)
].copy()

wdc_info_df["title"] = wdc_info_df["document_title"].map(remove_upload_text)
wdc_info_df["year"] = wdc_info_df["document_title"].map(extract_year)

def classify_wdc(title):
    t = title.lower()
    if "community project" in t or "proposed" in t or "project" in t:
        return "community project"
    if "stakeholder" in t or "minutes" in t:
        return "stakeholder/community meeting"
    if "cdf" in t:
        return "CDF"
    return "development information"

def identify_constituency(title):
    t = title.lower()
    if "mumbwa" in t and "nangoma" in t:
        return "Mumbwa Central and Nangoma"
    if "nangoma" in t:
        return "Nangoma"
    if "mumbwa" in t:
        return "Mumbwa Central"
    return ""

wdc_info_df["information_type"] = wdc_info_df["title"].map(classify_wdc)
wdc_info_df["constituency"] = wdc_info_df["title"].map(identify_constituency)
wdc_info_df["council_name"] = COUNCIL_NAME

wdc_info_df = wdc_info_df[
    ["council_name", "year", "information_type", "constituency", "title", "document_url"]
].rename(columns={"document_url": "source_url"})

wdc_info_df = (
    wdc_info_df
    .drop_duplicates(subset=["title", "source_url"])
    .sort_values(["year", "title"], na_position="last")
    .reset_index(drop=True)
)

#Public Council Resolutions/reports
resolution_sources = publication_df.copy()
resolution_sources["document_title"] = resolution_sources["document_title"].map(clean_text)
resolution_sources["document_url"] = resolution_sources["document_url"].map(clean_text)

resolution_sources = resolution_sources[
    resolution_sources["document_title"].str.contains(
        r"\bminutes\b|\bresolution\b",
        case=False,
        na=False,
        regex=True
    )
    & resolution_sources["document_url"].str.contains(
        r"\.pdf($|\?)",
        case=False,
        na=False,
        regex=True
    )
].drop_duplicates(subset=["document_url"])

resolution_keywords = re.compile(
    r"\b(resolved|resolution|agreed|approved|adopted|recommended|decision)\b",
    flags=re.I
)

resolution_records = []

for _, row in resolution_sources.iterrows():
    title = remove_upload_text(row["document_title"])
    url = row["document_url"]

    try:
        r = requests.get(url, timeout=45, verify=False)
        r.raise_for_status()

        with pdfplumber.open(BytesIO(r.content)) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text() or ""
                page_text = page_text.replace("\r", "\n")

                candidates = [
                    clean_text(line)
                    for line in page_text.split("\n")
                    if resolution_keywords.search(line or "")
                ]

                for statement in candidates:
                    if len(statement) >= 20:
                        resolution_records.append({
                            "council_name": COUNCIL_NAME,
                            "year": extract_year(title),
                            "source_document": title,
                            "page_number": page_number,
                            "resolution_text": statement,
                            "source_url": url
                        })

    except Exception as e:
        print(f"Could not process resolution source: {title}")
        print("Reason:", e)

public_resolutions_df = pd.DataFrame(
    resolution_records,
    columns=[
        "council_name",
        "year",
        "source_document",
        "page_number",
        "resolution_text",
        "source_url"
    ]
)

if not public_resolutions_df.empty:
    public_resolutions_df["resolution_text"] = public_resolutions_df["resolution_text"].map(clean_text)
    public_resolutions_df = (
        public_resolutions_df
        .drop_duplicates(subset=["source_document", "page_number", "resolution_text"])
        .reset_index(drop=True)
    )

# Export to cvs and preview
council_admin_df.to_csv("council administrative data.csv", sep="|", index=False, encoding="utf-8-sig")
wdc_info_df.to_csv("ward development committee information.csv", sep="|", index=False, encoding="utf-8-sig")
public_resolutions_df.to_csv("public council resolutions.csv", sep="|", index=False, encoding="utf-8-sig")

print("Created:")
print(f"1. council administrative data.csv -> {len(council_admin_df)} rows")
print(f"2. ward development committee information.csv -> {len(wdc_info_df)} rows")
print(f"3. public council resolutions.csv -> {len(public_resolutions_df)} rows")

print("\nAdministrative data preview:")
display(council_admin_df.head())

print("\nWDC information preview:")
display(wdc_info_df.head())

print("\nPublic resolutions preview:")
display(public_resolutions_df.head())


/tmp/ipykernel_2175/3188244587.py:174: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  wdc_info_df["document_url"].str.contains(r"\.pdf($|\?)", case=False, na=False, regex=True)
/tmp/ipykernel_2175/3188244587.py:227: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & resolution_sources["document_url"].str.contains(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate 

Created:
1. council administrative data.csv -> 16 rows
2. ward development committee information.csv -> 10 rows
3. public council resolutions.csv -> 119 rows

Administrative data preview:


,council_name,record_type,category,description,source_url
0,Mumbwa Town Council,department,Institutional Management,"Consists of the principal officer, the Council Secretary, as well as procurement and audit sections.",https://www.mumbwacouncil.gov.zm/?page_id=770
1,Mumbwa Town Council,department,Planning,"Plans the overall layout of the district, upgrading of shanty areas, township boundaries, standard building plans, land distribution, title deeds, HIV/AIDS, gender and human rights sensitisation, community development activities, and public health.",https://www.mumbwacouncil.gov.zm/?page_id=770
2,Mumbwa Town Council,department,Engineering Services,"Facilitates feeder roads, borehole drilling, inspection of buildings, scrutiny of building plans, burial sites, fire brigade services, and maintenance of plant and machinery.",https://www.mumbwacouncil.gov.zm/?page_id=770
3,Mumbwa Town Council,department,Finance,"Collection of revenue, preparation of statutory obligations, and preparation of books of accounts.",https://www.mumbwacouncil.gov.zm/?page_id=770
4,Mumbwa Town Council,department,Human Resource and Administration,"Staff establishment, asset management, records keeping, training and development, recruitment, staff appraisals, minutes and report writing, discipline, conditions of service, and public relations services.",https://www.mumbwacouncil.gov.zm/?page_id=770



WDC information preview:


,council_name,year,information_type,constituency,title,source_url
0,Mumbwa Town Council,2018,development information,,The-Constituency-Development-Fund-Act-No.-11-of-2018,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/The-Constituency-Development-Fund-Act-No.-11-of-2018.pdf
1,Mumbwa Town Council,2022,CDF,,CDF-Report-2022,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/CDF-Report-2022.pdf
2,Mumbwa Town Council,2023,CDF,Mumbwa Central,Press Statement Mumbwa CDF,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/Press-Statement-Mumbwa-CDF-Nov.pdf
3,Mumbwa Town Council,2024,community project,,2024 Approved CDF Community Projects,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2024/09/2024-CDF-COMMUNITY-PROJECTS-AS-AT-31ST-AUGUST-2024.pdf
4,Mumbwa Town Council,2024,stakeholder/community meeting,,2024 Minutes for the Stakeholders Engagement Meeting With Business Community,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/11/Minutes-for-the-Skaeholders-Engagement-Meeting-With-Business-Community-Held-on-5-July-2024.pdf



Public resolutions preview:


,council_name,year,source_document,page_number,resolution_text,source_url
0,Mumbwa Town Council,2023,"MINUTES OF THIRD ORDINARY COUNCIL MEETING OF 9TH OCTOBER, 2023",8,would be resolved quickly by relevant authorities.,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MINUTES-OF-THIRD-ORDINARY-COUNCIL-MEETING-OF-9TH-OCTOBER-2023.pdf
1,Mumbwa Town Council,2023,"MINUTES OF THIRD ORDINARY COUNCIL MEETING OF 9TH OCTOBER, 2023",9,2023 be Received and Adopted as part of,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MINUTES-OF-THIRD-ORDINARY-COUNCIL-MEETING-OF-9TH-OCTOBER-2023.pdf
2,Mumbwa Town Council,2023,"MINUTES OF THIRD ORDINARY COUNCIL MEETING OF 9TH OCTOBER, 2023",10,Received and Adopted as part of the proceedings of,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MINUTES-OF-THIRD-ORDINARY-COUNCIL-MEETING-OF-9TH-OCTOBER-2023.pdf
3,Mumbwa Town Council,2023,"MINUTES OF THIRD ORDINARY COUNCIL MEETING OF 9TH OCTOBER, 2023",10,Adopted as part of the proceedings of the,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MINUTES-OF-THIRD-ORDINARY-COUNCIL-MEETING-OF-9TH-OCTOBER-2023.pdf
4,Mumbwa Town Council,2023,"MINUTES OF THIRD ORDINARY COUNCIL MEETING OF 9TH OCTOBER, 2023",11,Received and Adopted as part of the proceedings of,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2023/12/MINUTES-OF-THIRD-ORDINARY-COUNCIL-MEETING-OF-9TH-OCTOBER-2023.pdf
